# 01 — Introductory Data Analysis (Phase 1)

**Goal:** practise the core toolkit — NumPy, Pandas, Matplotlib/Seaborn/Plotly, SQL and basic statistics — on a small, real, licensed dataset before we pick the main business dataset in Phase 2.

**Dataset:** UCI *Wine* (178 wines, 13 chemical measurements, 3 cultivars). Bundled with scikit-learn, licensed CC BY 4.0. Source: https://archive.ics.uci.edu/dataset/109/wine

> Rule for this project: every number written in markdown must come from a cell output above it. Run **Kernel → Restart & Run All** before committing.

In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from scipy import stats

from edip.data import load_wine_frame, run_query, to_sqlite
from edip.stats import mean_confidence_interval

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

## 1. NumPy — vectorised maths
NumPy applies an operation to a whole array at once (implemented in C), which is far faster and clearer than Python loops.

In [ ]:
a = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print("mean:", a.mean(), "| sample std (ddof=1):", round(a.std(ddof=1), 3))
print("z-scores:", np.round((a - a.mean()) / a.std(ddof=1), 2))   # broadcasting: scalar applied to every element
print("values above mean:", a[a > a.mean()])                      # boolean mask

## 2. Pandas — load and inspect

In [ ]:
df = load_wine_frame()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Data quality first: missing values and duplicates
print("missing values:", int(df.isna().sum().sum()))
print("duplicate rows:", int(df.duplicated().sum()))
df["cultivar"].value_counts()

In [ ]:
df.describe().T[["mean", "std", "min", "50%", "max"]]

## 3. Group comparisons with `groupby`
Equivalent to SQL `GROUP BY`.

In [ ]:
summary = df.groupby("cultivar").agg(
    n=("alcohol", "size"),
    alcohol_mean=("alcohol", "mean"),
    color_mean=("color_intensity", "mean"),
    proline_mean=("proline", "mean"),
)
summary

## 4. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x="alcohol", hue="cultivar", kde=True, ax=axes[0])
axes[0].set_title("Distribution of alcohol by cultivar")
sns.boxplot(data=df, x="cultivar", y="color_intensity", ax=axes[1])
axes[1].set_title("Colour intensity by cultivar")
plt.tight_layout()
plt.show()

In [ ]:
corr = df.drop(columns="cultivar").corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="vlag", center=0, square=True, cbar_kws={"label": "Pearson r"})
plt.title("Correlation matrix of chemical measurements")
plt.show()

In [ ]:
# Interactive chart — hover over points
px.scatter(df, x="flavanoids", y="proline", color="cultivar",
           title="Flavanoids vs proline (interactive)")

## 5. SQL with SQLite
We load the DataFrame into an in-memory database and query it. `run_query` only allows SELECT/WITH — a habit we'll enforce for AI agents in Phase 8.

In [ ]:
conn = sqlite3.connect(":memory:")
to_sqlite(df, "wine", conn)

run_query("""
SELECT cultivar,
       COUNT(*)               AS n_wines,
       ROUND(AVG(alcohol), 2) AS avg_alcohol
FROM wine
GROUP BY cultivar
HAVING COUNT(*) > 50
ORDER BY avg_alcohol DESC
""", conn)

In [ ]:
# Window function: each wine's alcohol rank inside its cultivar
run_query("""
SELECT cultivar, alcohol,
       RANK() OVER (PARTITION BY cultivar ORDER BY alcohol DESC) AS rank_in_group
FROM wine
ORDER BY cultivar, rank_in_group
LIMIT 6
""", conn)

## 6. Statistics — confidence interval and a hypothesis test

**95% confidence interval:** mean ± t* × s/√n. If we repeated the sampling many times, about 95% of intervals built this way would contain the true mean.

**Welch's t-test** asks whether two group means differ more than chance would explain, without assuming equal variances.
- H₀: mean alcohol of cultivar_0 = mean alcohol of cultivar_1
- A small p-value means the data would be unlikely if H₀ were true. It does **not** measure how big or important the difference is, and it says nothing about *why* (no causation).

In [ ]:
for name, group in df.groupby("cultivar"):
    low, high = mean_confidence_interval(group["alcohol"].tolist())
    print(f"{name}: mean={group['alcohol'].mean():.2f}, 95% CI=({low:.2f}, {high:.2f}), n={len(group)}")

In [ ]:
g0 = df.loc[df["cultivar"] == "cultivar_0", "alcohol"]
g1 = df.loc[df["cultivar"] == "cultivar_1", "alcohol"]
t_stat, p_value = stats.ttest_ind(g0, g1, equal_var=False)

# Effect size (Cohen's d) — how large the difference is in standard-deviation units
pooled_sd = np.sqrt((g0.var(ddof=1) + g1.var(ddof=1)) / 2)
cohens_d = (g0.mean() - g1.mean()) / pooled_sd
print(f"t = {t_stat:.2f}, p = {p_value:.2e}, Cohen's d = {cohens_d:.2f}")

**Interpret in your own words below** (use the numbers printed above, not memory):

- *What does the p-value tell us?* …
- *Is the difference large in practical terms (Cohen's d)?* …
- *Does this show cultivar **causes** higher alcohol?* …

## 7. Exercises (complete before Phase 2)

1. **NumPy:** without `np.std`, compute the sample standard deviation of `df["proline"].to_numpy()` and compare with `np.std(..., ddof=1)`.
2. **Pandas:** add a column `alcohol_band` = `"high"` if alcohol ≥ 13 else `"normal"`, then use `pd.crosstab` against `cultivar`.
3. **Visualisation:** make one Plotly box plot of `proline` by cultivar. Write two sentences on what it shows.
4. **SQL:** return the top 3 wines by alcohol *per cultivar* (hint: CTE + window function).
5. **Statistics:** compute a 95% CI for `color_intensity` in each cultivar. Do the intervals overlap?
6. **Correlation vs causation:** pick the strongest correlation in the heatmap and explain one reason it might **not** be causal.
7. **Git:** commit your solutions on a branch with message `feat(notebook): complete phase 1 exercises` and open a PR.

In [ ]:
# Exercise workspace


In [ ]:
conn.close()